# 공차 전체 음료 칼로리 분석

- **수집 대상:** https://www.gong-cha.co.kr/brand/menu/product?category=001001
- **robots.txt 확인:** `/brand/menu/` 경로는 수집 허용 (`disallow: /thdadmin/, /upload/, /html2/, /html/`)
- **목적:** 공차 음료 중 칼로리가 가장 높은/낮은 음료 Top 5 찾기 (**L 사이즈 아이스(Cold) 기준**)

### 사이트 구조 (직접 확인한 내용)
- 음료 탭(New 시즌 메뉴, 밀크티, 커피 …)은 클릭하면 주소의 `category` 번호만 바뀌는 **정적 페이지**예요.
  → Selenium 없이 `requests` + `BeautifulSoup`으로 충분해요.
- 상세 페이지 영양정보 표의 열 순서: `구분(Cold/Hot) | 사이즈(L/J) | 컵 용량(ml) | 열량(kcal) | 당류 | 단백질 | 포화지방 | 나트륨 | 카페인`

**셀을 위에서부터 순서대로 실행하세요** (`Shift + Enter`).

## 1. 라이브러리 불러오기 + 설정

In [1]:
import re
import time

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE = "https://www.gong-cha.co.kr"
LIST_URL = BASE + "/brand/menu/product?category={code}"
HEADERS = {"User-Agent": "Mozilla/5.0"}

# 음료 탭 이름과 category 번호 (사이트의 탭 HTML에서 확인한 값)
# 먼저 나온 탭 이름이 그 음료의 카테고리가 되도록 진짜 종류를 앞에 뒀어요.
TABS = {
    "001006": "밀크티",
    "001010": "스무디",
    "001003": "오리지널 티",
    "001015": "프룻티&모어",
    "001011": "커피",
    "001017": "요거티",
    "001001": "New 시즌 메뉴",
    "001002": "베스트셀러",
}

## 2. 함수: 탭(카테고리) 페이지에서 메뉴 이름과 상세 링크 모으기

In [2]:
def get_menus_in_tab(code):
    res = requests.get(LIST_URL.format(code=code), headers=HEADERS, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")

    items = []
    # 메뉴 하나 = <a href="/brand/menu/product_detail?category=...&no=..."> 안의 <p>메뉴이름</p>
    for a in soup.select('a[href*="product_detail"]'):
        name = " ".join(a.get_text(" ", strip=True).split())
        url = BASE + a["href"]               # 링크 주소를 그대로 사용 (category 번호 포함)
        if name:
            items.append({"name": name, "url": url})
    return items

## 3. 함수: 상세 페이지 영양정보 표에서 L 사이즈 아이스(Cold) 값 가져오기

In [3]:
def get_nutrition(url):
    res = requests.get(url, headers=HEADERS, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")

    rows = []
    temp = None                                  # Cold / Hot
    for tr in soup.select("table tbody tr"):
        cells = [td.get_text(strip=True) for td in tr.find_all("td")]
        if len(cells) < 8:
            continue
        # 표에서 'Cold'/'Hot' 칸은 여러 줄을 합쳐 놓아서(rowspan) 첫 줄에만 있어요.
        # 그래서 칸이 9개면 첫 칸이 온도, 8개면 위 줄의 온도를 그대로 써요.
        if len(cells) == 9:
            temp = cells[0]
        # 뒤에서부터 세면 칸 위치가 항상 같아요:
        # [-8]사이즈 [-7]용량 [-6]열량 [-5]당류 [-4]단백질 [-3]포화지방 [-2]나트륨 [-1]카페인
        rows.append({
            "온도": temp,
            "사이즈": cells[-8],
            "용량(ml)": float(cells[-7]),
            "칼로리(kcal)": float(cells[-6]),
            "당류(g)": float(cells[-5]),
            "카페인(mg)": float(cells[-1]),
        })

    # L 사이즈 아이스(Cold) 줄 찾기 → 없으면 None
    for r in rows:
        if r["온도"] == "Cold" and r["사이즈"] == "L":
            return r
    return None

## 4. 1단계 실행: 모든 탭에서 음료 모으기 (중복 제거)

In [4]:
menus = {}                                   # {메뉴이름: 정보}  → 이름이 같으면 한 번만
for code, tab in TABS.items():
    items = get_menus_in_tab(code)
    new = 0
    for it in items:
        if it["name"] not in menus:
            menus[it["name"]] = {"name": it["name"], "category": tab, "url": it["url"]}
            new += 1
    print(f"[{tab}] {len(items)}개 (새로 추가 {new}개)")
    time.sleep(1)                            # 서버 부담 줄이기

menus = list(menus.values())
print(f"\n중복 빼고 음료 {len(menus)}개")

[밀크티] 12개 (새로 추가 12개)
[스무디] 17개 (새로 추가 17개)
[오리지널 티] 4개 (새로 추가 4개)
[프룻티&모어] 9개 (새로 추가 9개)
[커피] 6개 (새로 추가 6개)
[요거티] 4개 (새로 추가 4개)
[New 시즌 메뉴] 36개 (새로 추가 36개)
[베스트셀러] 14개 (새로 추가 13개)

중복 빼고 음료 101개


## 5. 2단계 실행: 각 음료의 영양정보 모으기
음료 하나당 1초씩 쉬어서 2분 정도 걸려요.

In [5]:
for i, m in enumerate(menus, 1):
    try:
        info = get_nutrition(m["url"])
    except Exception as e:
        print("  에러:", m["name"], e)
        info = None
    if info:
        m.update(info)
        print(f"[{i}/{len(menus)}] {m['name']}: {info['칼로리(kcal)']:.0f}kcal")
    else:
        print(f"[{i}/{len(menus)}] {m['name']}: L 사이즈 아이스 정보 없음 (제외)")
    time.sleep(1)

[1/101] 브라운슈가 시그니처 밀크티 +미니펄: 407kcal
[2/101] 미니펄 망고 밀크: 301kcal
[3/101] 미니펄 딸기 밀크티: 296kcal
[4/101] 피스타치오 밀크티: 227kcal
[5/101] 제주 그린 밀크티: 184kcal
[6/101] 브라운슈가 쥬얼리 밀크티: 328kcal
[7/101] 얼그레이 밀크티: 205kcal
[8/101] 자스민 밀크티: 206kcal
[9/101] 우롱 밀크티: 205kcal
[10/101] 블랙 밀크티: 206kcal
[11/101] 타로 밀크티: 263kcal
[12/101] 초콜렛 밀크티: 196kcal
[13/101] 타로 쥬얼리 시그니처 스무디: 467kcal
[14/101] 초코바른 초코 스무디: 606kcal
[15/101] 초코바른 제주 그린 스무디: 498kcal
[16/101] 초코바른 피스타치오 스무디: 476kcal
[17/101] 밀크 쿠앤크 스무디: 459kcal
[18/101] 브라운슈가 쥬얼리 치즈폼 스무디: 572kcal
[19/101] 미니펄 망고 크러쉬: 408kcal
[20/101] 미니펄 딸기 크러쉬: 466kcal
[21/101] 타로 스무디: 354kcal
[22/101] 레몬 요구르트 스무디: 343kcal
[23/101] 초코멜로 스무디: 460kcal
[24/101] 초콜렛 쿠키 & 크림 스무디: 516kcal
[25/101] 딸기 쿠키 스무디: 587kcal
[26/101] 딸기 쥬얼리 요구르트 크러쉬: 349kcal
[27/101] 제주 그린 스무디: 476kcal
[28/101] 망고 스무디: 407kcal
[29/101] 청포도 스무디: 329kcal
[30/101] 자스민티: 4kcal
[31/101] 우롱티: 3kcal
[32/101] 블랙티: 4kcal
[33/101] 얼그레이티: 3kcal
[34/101] 허니 자몽 블랙티: 126kcal
[35/101] 레몬 자스민티: 112kcal
[36/101] 레몬 요구르트: 212kcal

## 6. 데이터 확인 + CSV 저장

In [6]:
df = pd.DataFrame(menus)
df.to_csv("gongcha_all_drinks.csv", index=False, encoding="utf-8-sig")  # 엑셀에서 한글 안 깨지게
print("CSV 저장 완료: gongcha_all_drinks.csv")
df

CSV 저장 완료: gongcha_all_drinks.csv


,name,category,url,온도,사이즈,용량(ml),칼로리(kcal),당류(g),카페인(mg)
0,브라운슈가 시그니처 밀크티 +미니펄,밀크티,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,407.0,40.0,92.0
1,미니펄 망고 밀크,밀크티,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,301.0,33.0,0.0
2,미니펄 딸기 밀크티,밀크티,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,296.0,40.0,49.0
3,피스타치오 밀크티,밀크티,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,227.0,25.0,20.0
4,제주 그린 밀크티,밀크티,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,184.0,16.0,49.0
...,...,...,...,...,...,...,...,...,...
96,우롱티 + 코코넛 + 밀크폼,베스트셀러,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,137.0,13.0,64.0
97,딸기 쥬얼리 밀크티,베스트셀러,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,225.0,26.0,29.0
98,망고 요구르트 + 화이트펄,베스트셀러,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,209.0,36.0,0.0
99,타로 밀크티 + 펄,베스트셀러,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,357.0,14.0,0.0


## 7. 분석: 칼로리 높은/낮은 음료 Top 5 (L 사이즈 아이스 기준)

In [7]:
data = df.dropna(subset=["칼로리(kcal)"])   # L 사이즈 아이스 정보가 없는 음료는 제외
top5 = data.sort_values("칼로리(kcal)", ascending=False, kind="stable").head(5)
low5 = data.sort_values("칼로리(kcal)", ascending=True, kind="stable").head(5)

print("[칼로리 높은 음료 Top 5] (L 사이즈 아이스 기준)")
for rank, (name, kcal) in enumerate(zip(top5["name"], top5["칼로리(kcal)"]), 1):
    print(f"{rank}등. {name} {kcal:.0f}kcal")

print("\n[칼로리 낮은 음료 Top 5] (L 사이즈 아이스 기준)")
for rank, (name, kcal) in enumerate(zip(low5["name"], low5["칼로리(kcal)"]), 1):
    print(f"{rank}등. {name} {kcal:.0f}kcal")

print(f"\n분석한 음료 {len(data)}개, 평균 칼로리 {data['칼로리(kcal)'].mean():.0f}kcal")

[칼로리 높은 음료 Top 5] (L 사이즈 아이스 기준)
1등. 딸기 쿠키 스무디 + 펄 679kcal
2등. 초코바른 초코 스무디 606kcal
3등. 딸기 쿠키 스무디 587kcal
4등. 브라운슈가 쥬얼리 치즈폼 스무디 572kcal
5등. 초콜릿 시그니처 밀크티+펄 540kcal

[칼로리 낮은 음료 Top 5] (L 사이즈 아이스 기준)
1등. 우롱티 3kcal
2등. 얼그레이티 3kcal
3등. 자스민티 4kcal
4등. 블랙티 4kcal
5등. 저당 우롱 오리지널 티 5kcal

분석한 음료 98개, 평균 칼로리 275kcal


In [8]:
# 8. 순위 출력 함수 (값이 같으면 같은 등수)
def print_rank(table, col, unit, n=5, title="", ascending=False):
    """table에서 col 값 순서로 n등까지 출력 (동점은 같은 등수)
    ascending=False → 큰 값부터(많은 순), ascending=True → 작은 값부터(적은 순)"""
    t = table.dropna(subset=[col]).copy()
    t["순위"] = t[col].rank(method="min", ascending=ascending).astype(int)   # 동점 = 같은 등수
    t = t[t["순위"] <= n].sort_values(["순위", "name"])
    print(title)
    for _, row in t.iterrows():
        print(f"{row['순위']}등. {row['name']} {row[col]:.0f}{unit}")

In [9]:
# 9-1. 당류가 가장 많은 음료 Top 5
print_rank(data, "당류(g)", "g", n=5, title="[당류 많은 음료 Top 5] (L 사이즈 아이스 기준)")

# 9-2. 당류가 0g인 음료 목록
no_sugar = data[data["당류(g)"] == 0]
print(f"\n[당류 0g 음료] {len(no_sugar)}개 / 전체 {len(data)}개")
for i, name in enumerate(no_sugar["name"], 1):
    print(f"{i}. {name}")

[당류 많은 음료 Top 5] (L 사이즈 아이스 기준)
1등. 청포도 스무디 69g
2등. 레몬 요구르트 스무디 66g
3등. 얼그레이 말차 밀크티 + 펄 63g
4등. 딸기 쿠키 스무디 + 펄 58g
5등. 포도 말랑 크러쉬 57g

[당류 0g 음료] 10개 / 전체 98개
1. 자스민티
2. 우롱티
3. 블랙티
4. 얼그레이티
5. 얼그레이 아메리카노
6. 저당 블랙 오리지널 티
7. 저당 얼그레이 오리지널 티
8. 저당 우롱 오리지널 티
9. 저당 자스민 오리지널 티
10. 아메리카노


In [10]:
# 10. 카페인이 가장 많은 음료 Top 5 + 카페인 없는 음료
print_rank(data, "카페인(mg)", "mg", n=5, title="[카페인 많은 음료 Top 5] (L 사이즈 아이스 기준)")

no_caffeine = data[data["카페인(mg)"] == 0]
print(f"\n카페인 0mg 음료: {len(no_caffeine)}개 / 전체 {len(data)}개")
print(", ".join(no_caffeine["name"]))

[카페인 많은 음료 Top 5] (L 사이즈 아이스 기준)
1등. 얼그레이 말차 밀크티 + 펄 186mg
2등. 공차슈페너 178mg
3등. 얼그레이 밀크티 175mg
3등. 저당 얼그레이 밀크티 175mg
3등. 저당 얼그레이 밀크티 + 펄 175mg
3등. 포도 말랑 밀크티 175mg

카페인 0mg 음료: 21개 / 전체 98개
미니펄 망고 밀크, 브라운슈가 쥬얼리 밀크티, 타로 밀크티, 타로 쥬얼리 시그니처 스무디, 밀크 쿠앤크 스무디, 브라운슈가 쥬얼리 치즈폼 스무디, 미니펄 망고 크러쉬, 미니펄 딸기 크러쉬, 타로 스무디, 레몬 요구르트 스무디, 망고 스무디, 레몬 요구르트, 자몽 요구르트, 패션 프룻 히비스커스, 망고 주스, 망고 요구르트, 코코 타로 쥬얼리 밀크티, 타로 쥬얼리 시그니처 밀크티, 미니펄 망고 밀크 + 리얼 망고, 망고 요구르트 + 화이트펄, 타로 밀크티 + 펄


In [11]:
# 11. 베스트셀러 중 칼로리 Top 3
best_names = [it["name"] for it in get_menus_in_tab("001002")]   # 베스트셀러 탭 메뉴 이름
best = data[data["name"].isin(best_names)]
print(f"베스트셀러 {len(best_names)}개 중 분석 가능한 음료 {len(best)}개\n")

# 11-1. 베스트셀러 중 칼로리 높은 Top 3
print_rank(best, "칼로리(kcal)", "kcal", n=3, title="[베스트셀러 칼로리 높은 Top 3] (L 사이즈 아이스 기준)")

# 11-2. 베스트셀러 중 칼로리 낮은 Top 3
print_rank(best, "칼로리(kcal)", "kcal", n=3, ascending=True,
           title="\n[베스트셀러 칼로리 낮은 Top 3] (L 사이즈 아이스 기준)")

베스트셀러 14개 중 분석 가능한 음료 14개

[베스트셀러 칼로리 높은 Top 3] (L 사이즈 아이스 기준)
1등. 딸기 쿠키 스무디 + 펄 679kcal
2등. 브라운슈가 시그니처 밀크티 +펄 424kcal
3등. 딸기 쥬얼리 시그니처 밀크티 386kcal

[베스트셀러 칼로리 낮은 Top 3] (L 사이즈 아이스 기준)
1등. 아메리카노 10kcal
2등. 허니 자몽 블랙티 126kcal
3등. 우롱티 + 코코넛 + 밀크폼 137kcal


In [12]:
# 카테고리 별 평균 비교
rows = []
for code, tab in TABS.items():
    names = [it["name"] for it in get_menus_in_tab(code)]   # 이 탭에 있는 메뉴 이름
    t = data[data["name"].isin(names)]                       # 그중 분석 가능한(L 아이스) 음료
    rows.append({
        "카테고리": tab,
        "음료 수": len(t),
        "평균 칼로리(kcal)": round(t["칼로리(kcal)"].mean(), 1),
        "평균 당류(g)": round(t["당류(g)"].mean(), 1),
        "평균 카페인(mg)": round(t["카페인(mg)"].mean(), 1),
    })
    time.sleep(1)

cat_df = pd.DataFrame(rows)

# 진짜 음료 종류만 비교 (New 시즌 메뉴, 베스트셀러는 여러 종류가 섞여 있어서 따로 표시)
kind = cat_df[~cat_df["카테고리"].isin(["New 시즌 메뉴", "베스트셀러"])]
mixed = cat_df[cat_df["카테고리"].isin(["New 시즌 메뉴", "베스트셀러"])]

for col, unit in [("평균 칼로리(kcal)", "kcal"), ("평균 당류(g)", "g"), ("평균 카페인(mg)", "mg")]:
    print(f"[{col.split('(')[0]} 높은 카테고리 순]")
    for i, (_, r) in enumerate(kind.sort_values(col, ascending=False).iterrows(), 1):
        print(f"{i}등. {r['카테고리']} {r[col]}{unit} (음료 {r['음료 수']}개)")
    print()

print("[참고: 여러 종류가 섞인 탭]")
print(mixed.to_string(index=False))

[평균 칼로리 높은 카테고리 순]
1등. 스무디 457.2kcal (음료 17개)
2등. 요거티 283.8kcal (음료 4개)
3등. 밀크티 252.0kcal (음료 12개)
4등. 커피 188.0kcal (음료 6개)
5등. 프룻티&모어 126.7kcal (음료 9개)
6등. 오리지널 티 3.5kcal (음료 4개)

[평균 당류 높은 카테고리 순]
1등. 스무디 49.9g (음료 17개)
2등. 요거티 35.0g (음료 4개)
3등. 프룻티&모어 25.4g (음료 9개)
4등. 밀크티 17.3g (음료 12개)
5등. 커피 15.5g (음료 6개)
6등. 오리지널 티 0.0g (음료 4개)

[평균 카페인 높은 카테고리 순]
1등. 커피 147.0mg (음료 6개)
2등. 오리지널 티 97.8mg (음료 4개)
3등. 밀크티 58.0mg (음료 12개)
4등. 요거티 41.5mg (음료 4개)
5등. 프룻티&모어 36.8mg (음료 9개)
6등. 스무디 27.8mg (음료 17개)

[참고: 여러 종류가 섞인 탭]
     카테고리  음료 수  평균 칼로리(kcal)  평균 당류(g)  평균 카페인(mg)
New 시즌 메뉴    33         265.4      21.1        78.4
    베스트셀러    14         291.4      25.0        58.1


In [13]:
# 13. 저당 메뉴 vs 일반 메뉴
def norm(s):
    """띄어쓰기를 지워서 '+ 펄'과 '+펄'처럼 띄어쓰기만 다른 이름도 같게 보기"""
    return re.sub(r"\s", "", s)

by_name = {norm(n): row for n, row in zip(data["name"], data.to_dict("records"))}

pairs, no_pair = [], []
for _, low in data[data["name"].str.startswith("저당")].iterrows():
    base = low["name"].replace("저당", "", 1).strip()          # '저당 블랙 밀크티' → '블랙 밀크티'
    normal = by_name.get(norm(base))
    if normal is None:                                         # '저당 블랙 오리지널 티' → '블랙티'
        m = re.match(r"^(.*?)\s*오리지널\s*티$", base)
        if m:
            normal = by_name.get(norm(m.group(1) + "티"))
    if normal is None:
        no_pair.append(low["name"])
        continue
    pairs.append({
        "저당 메뉴": low["name"], "일반 메뉴": normal["name"],
        "저당 칼로리": low["칼로리(kcal)"], "일반 칼로리": normal["칼로리(kcal)"],
        "칼로리 차이": low["칼로리(kcal)"] - normal["칼로리(kcal)"],
        "저당 당류": low["당류(g)"], "일반 당류": normal["당류(g)"],
        "당류 차이": low["당류(g)"] - normal["당류(g)"],
    })

pair_df = pd.DataFrame(pairs)
print(f"짝을 찾은 저당 메뉴 {len(pair_df)}개, 짝이 없는 저당 메뉴 {len(no_pair)}개")
print("짝 없음:", ", ".join(no_pair))
print(f"\n평균 칼로리 차이(저당 - 일반): {pair_df['칼로리 차이'].mean():+.1f}kcal")
print(f"평균 당류 차이(저당 - 일반): {pair_df['당류 차이'].mean():+.1f}g\n")
print(pair_df.to_string(index=False))

짝을 찾은 저당 메뉴 11개, 짝이 없는 저당 메뉴 4개
짝 없음: 저당 햅쌀 미숫가루 밀크티 + 펄, 저당 얼그레이 밀크티 + 펄, 저당 우롱 밀크티 + 펄, 저당 자스민 밀크티 + 펄

평균 칼로리 차이(저당 - 일반): +2.9kcal
평균 당류 차이(저당 - 일반): +0.0g

           저당 메뉴         일반 메뉴  저당 칼로리  일반 칼로리  칼로리 차이  저당 당류  일반 당류  당류 차이
       저당 블랙 밀크티        블랙 밀크티   209.0   206.0     3.0    2.0    2.0    0.0
     저당 얼그레이 밀크티      얼그레이 밀크티   209.0   205.0     4.0    2.0    2.0    0.0
       저당 우롱 밀크티        우롱 밀크티   208.0   205.0     3.0    2.0    2.0    0.0
      저당 자스민 밀크티       자스민 밀크티   209.0   206.0     3.0    2.0    2.0    0.0
   저당 블랙 밀크티 + 펄    블랙 밀크티 + 펄   301.0   299.0     2.0    5.0    5.0    0.0
    저당 블랙 오리지널 티           블랙티     6.0     4.0     2.0    0.0    0.0    0.0
  저당 얼그레이 오리지널 티         얼그레이티     6.0     3.0     3.0    0.0    0.0    0.0
    저당 우롱 오리지널 티           우롱티     5.0     3.0     2.0    0.0    0.0    0.0
   저당 자스민 오리지널 티          자스민티     6.0     4.0     2.0    0.0    0.0    0.0
      저당 초콜렛 밀크티       초콜렛 밀크티   200.0   196.0     4.0    2.0    2.0    0.0
저당 

# 공차 음료 영양정보 EDA 보고서

## 1. 개요

| 항목 | 내용 |
|---|---|
| 주제 | 공차 음료의 칼로리·당류·카페인 분석 |
| 목적 | 가장 달고, 칼로리가 높고, 카페인이 없는 메뉴는 무엇인지 데이터로 확인 |
| 데이터 출처 | 공차 코리아 공식 홈페이지 음료 메뉴 페이지<br>https://www.gong-cha.co.kr/brand/menu/product?category=001001 |
| 기준 | **L 사이즈 아이스(Cold, 473ml)** |

---

## 2. 데이터 수집

### 2-1. 수집 가능 여부 확인 (robots.txt)

```
user-agent: *
disallow: /thdadmin/
disallow: /upload/
disallow: /html2/
disallow: /html/
```

- 수집한 경로 `/brand/menu/`는 금지 목록에 없어 **수집 허용** 경로임
- 이용약관에도 크롤링·자동 수집 금지 조항은 없음
- 학습 목적으로 텍스트·숫자 정보만 수집했고, 이미지는 수집하지 않음

### 2-2. 사이트 구조

- 음료 탭(밀크티, 스무디, 커피 등)은 주소의 `category` 번호만 바뀌는 **정적 페이지**
- 따라서 Selenium 없이 `requests` + `BeautifulSoup`으로 수집

| 탭 | category 번호 |
|---|---|
| New 시즌 메뉴 | 001001 |
| 베스트셀러 | 001002 |
| 오리지널 티 | 001003 |
| 밀크티 | 001006 |
| 스무디 | 001010 |
| 커피 | 001011 |
| 프룻티&모어 | 001015 |
| 요거티 | 001017 |

### 2-3. 수집 과정

1. 탭 8개의 목록 페이지에서 **메뉴 이름과 상세 페이지 링크** 수집
2. 각 상세 페이지의 영양정보 표에서 **L 사이즈 아이스** 행의 값 수집
3. 서버 부담을 줄이기 위해 요청마다 1초씩 대기

---

## 3. 데이터 설명

### 3-1. 수집 결과

| 탭 | 메뉴 수 | 새로 추가 |
|---|---|---|
| 밀크티 | 12 | 12 |
| 스무디 | 17 | 17 |
| 오리지널 티 | 4 | 4 |
| 프룻티&모어 | 9 | 9 |
| 커피 | 6 | 6 |
| 요거티 | 4 | 4 |
| New 시즌 메뉴 | 36 | 36 |
| 베스트셀러 | 14 | 13 |
| **합계** | **102** | **101** |

- 베스트셀러의 '허니 자몽 블랙티'가 프룻티&모어와 겹쳐서 **중복을 빼면 101개**
- 납작복숭아 아이스티 3종은 J 사이즈(651ml)만 있어서 L 기준 분석에서 제외
- **최종 분석 대상: 98개 음료**

### 3-2. 변수

| 변수 | 설명 | 단위 |
|---|---|---|
| name | 메뉴 이름 | - |
| category | 처음 수집된 탭 이름 | - |
| 칼로리 | 열량 | kcal |
| 당류 | 당류 | g |
| 카페인 | 카페인 | mg |

> 동점은 같은 등수로 표시함 (예: 3kcal인 음료가 2개면 둘 다 1등)

---

## 4. 분석 결과

### 4-1. 칼로리

**전체 98개 음료의 평균 칼로리: 275kcal**

| 칼로리 높은 Top 5 | | 칼로리 낮은 Top 5 | |
|---|---|---|---|
| 1등. 딸기 쿠키 스무디 + 펄 | 679kcal | 1등. 우롱티 | 3kcal |
| 2등. 초코바른 초코 스무디 | 606kcal | 1등. 얼그레이티 | 3kcal |
| 3등. 딸기 쿠키 스무디 | 587kcal | 3등. 자스민티 | 4kcal |
| 4등. 브라운슈가 쥬얼리 치즈폼 스무디 | 572kcal | 3등. 블랙티 | 4kcal |
| 5등. 초콜릿 시그니처 밀크티+펄 | 540kcal | 5등. 저당 우롱 오리지널 티 | 5kcal |

- 칼로리 높은 Top 5 중 **4개가 스무디**. 쿠키, 초콜릿, 치즈폼처럼 토핑이 많이 들어간 메뉴가 상위권
- 칼로리 낮은 Top 5는 **모두 오리지널 티** (우유·시럽 없이 차만 들어간 메뉴)
- 가장 높은 음료(679kcal)와 가장 낮은 음료(3kcal)의 차이는 **약 226배**
- '딸기 쿠키 스무디'에 펄을 추가하면 587 → 679kcal로 **92kcal 증가**

### 4-2. 당류

**당류 많은 Top 5**

| 순위 | 메뉴 | 당류 |
|---|---|---|
| 1등 | 청포도 스무디 | 69g |
| 2등 | 레몬 요구르트 스무디 | 66g |
| 3등 | 얼그레이 말차 밀크티 + 펄 | 63g |
| 4등 | 딸기 쿠키 스무디 + 펄 | 58g |
| 5등 | 포도 말랑 크러쉬 | 57g |

- Top 5 모두 **WHO 하루 당류 섭취 권고량 50g**(총열량의 10%)을 한 잔으로 넘음
- 과일 맛 스무디·크러쉬가 상위권. 칼로리 1등이 아닌 메뉴도 당류는 높을 수 있음

**당류 0g 음료 (10개 / 98개)**

| 종류 | 메뉴 |
|---|---|
| 오리지널 티 | 블랙티, 우롱티, 얼그레이티, 자스민티 |
| 저당 오리지널 티 | 저당 블랙·얼그레이·우롱·자스민 오리지널 티 |
| 커피 | 아메리카노, 얼그레이 아메리카노 |

- 당류 0g 음료는 모두 **차 또는 커피에 아무것도 넣지 않은 메뉴**
- 단 음료를 피하고 싶다면 오리지널 티나 아메리카노가 가장 확실한 선택

### 4-3. 카페인

**카페인 많은 Top 5**

| 순위 | 메뉴 | 카페인 |
|---|---|---|
| 1등 | 얼그레이 말차 밀크티 + 펄 | 186mg |
| 2등 | 공차슈페너 | 178mg |
| 3등 | 얼그레이 밀크티 | 175mg |
| 3등 | 저당 얼그레이 밀크티 | 175mg |
| 3등 | 저당 얼그레이 밀크티 + 펄 | 175mg |
| 3등 | 포도 말랑 밀크티 | 175mg |

- 3등이 4개로 공동이라 Top 5에 6개가 포함됨
- 상위권에 **얼그레이 베이스 밀크티**가 많음. 커피보다 차 기반 밀크티의 카페인이 높은 경우도 있음
- 1등 음료 두 잔(372mg)이면 식약처 성인 하루 카페인 최대 섭취 권고량(400mg)에 가까워짐

**카페인 0mg 음료 (21개 / 98개, 약 21%)**

| 종류 | 메뉴 |
|---|---|
| 타로 | 타로 밀크티, 타로 밀크티 + 펄, 타로 스무디, 타로 쥬얼리 시그니처 스무디, 타로 쥬얼리 시그니처 밀크티, 코코 타로 쥬얼리 밀크티 |
| 망고 | 망고 스무디, 망고 주스, 망고 요구르트, 망고 요구르트 + 화이트펄, 미니펄 망고 밀크, 미니펄 망고 밀크 + 리얼 망고, 미니펄 망고 크러쉬 |
| 요구르트·과일 | 레몬 요구르트, 레몬 요구르트 스무디, 자몽 요구르트, 미니펄 딸기 크러쉬, 패션 프룻 히비스커스 |
| 기타 | 브라운슈가 쥬얼리 밀크티, 브라운슈가 쥬얼리 치즈폼 스무디, 밀크 쿠앤크 스무디 |

- 이름에 '밀크티'가 들어가도 **타로 밀크티는 카페인 0mg** (차 대신 타로가 들어감)
- 밤에 마시기 좋은 음료는 **타로, 망고, 요구르트 계열**

### 4-4. 베스트셀러 칼로리 (14개)

| 칼로리 높은 Top 3 | | 칼로리 낮은 Top 3 | |
|---|---|---|---|
| 1등. 딸기 쿠키 스무디 + 펄 | 679kcal | 1등. 아메리카노 | 10kcal |
| 2등. 브라운슈가 시그니처 밀크티 +펄 | 424kcal | 2등. 허니 자몽 블랙티 | 126kcal |
| 3등. 딸기 쥬얼리 시그니처 밀크티 | 386kcal | 3등. 우롱티 + 코코넛 + 밀크폼 | 137kcal |

- 전체 칼로리 1등인 **딸기 쿠키 스무디 + 펄이 베스트셀러에도 포함**됨
- 베스트셀러 안에서도 아메리카노(10kcal)부터 679kcal까지 **차이가 매우 큼**
- 인기 메뉴라고 해서 칼로리가 비슷하지 않음. 고칼로리 디저트형 메뉴와 가벼운 차가 함께 인기

### 4-5. 카테고리별 비교

| 카테고리 | 음료 수 | 평균 칼로리 | 평균 당류 | 평균 카페인 |
|---|---|---|---|---|
| 스무디 | 17 | 457.2kcal | 49.9g | 27.8mg |
| 요거티 | 4 | 283.8kcal | 35.0g | 41.5mg |
| 밀크티 | 12 | 252.0kcal | 17.3g | 58.0mg |
| 커피 | 6 | 188.0kcal | 15.5g | 147.0mg |
| 프룻티&모어 | 9 | 126.7kcal | 25.4g | 36.8mg |
| 오리지널 티 | 4 | 3.5kcal | 0.0g | 97.8mg |
| (참고) 베스트셀러 | 14 | 291.4kcal | 25.0g | 58.1mg |
| (참고) New 시즌 메뉴 | 33 | 265.4kcal | 21.1g | 78.4mg |

- **스무디는 칼로리와 당류 모두 1위.** 평균 457kcal로 오리지널 티(3.5kcal)의 약 130배이고, 평균 당류 49.9g은 WHO 하루 권고량 50g과 거의 같음
- **카페인은 커피(147mg) > 오리지널 티(97.8mg) > 밀크티(58mg) 순.** 밀크티는 우유가 섞여 차만 마실 때보다 카페인이 낮음
- **프룻티&모어는 칼로리는 낮지만(126.7kcal) 당류는 밀크티보다 높음(25.4g).** 칼로리가 낮다고 덜 단 음료는 아님
- 베스트셀러의 평균 칼로리(291.4kcal)는 전체 평균(275kcal)보다 약간 높음
- New 시즌 메뉴와 베스트셀러는 여러 종류가 섞여 있어서 참고용으로만 표시함

### 4-6. 저당 메뉴 vs 일반 메뉴

| 저당 메뉴 | 칼로리 / 당류 | 일반 메뉴 | 칼로리 / 당류 |
|---|---|---|---|
| 저당 블랙 밀크티 | 209kcal / 2g | 블랙 밀크티 | 206kcal / 2g |
| 저당 우롱 밀크티 | 208kcal / 2g | 우롱 밀크티 | 205kcal / 2g |
| 저당 초콜렛 밀크티 | 200kcal / 2g | 초콜렛 밀크티 | 196kcal / 2g |
| 저당 블랙 오리지널 티 | 6kcal / 0g | 블랙티 | 4kcal / 0g |
| … (총 11쌍) | | | |

- 짝을 찾은 11쌍의 평균 차이는 **칼로리 +2.9kcal, 당류 0g**
- 짝이 없는 저당 메뉴 4개: 저당 자스민·우롱·얼그레이 밀크티 + 펄, 저당 햅쌀 미숫가루 밀크티 + 펄
- **홈페이지 수치로는 저당 메뉴와 일반 메뉴의 차이가 거의 없고, 칼로리는 오히려 저당 메뉴가 조금 높음**
- 하지만 저당 블랙 밀크티 상세 페이지에는 "기존 블랙 밀크티 추천 당도 대비 당류가 81% 감소"라고 적혀 있음
- 이 결과는 저당 메뉴의 효과가 없다는 뜻이 아니라, **일반 메뉴의 영양정보가 당도 시럽을 넣지 않은 값(당도 0%)이기 때문으로 추정됨** (6장 한계점 참고)

---

## 5. 결론

## 5. 결론

1. **고칼로리·고당류 메뉴는 스무디에 몰려 있음.**
   칼로리 Top 5 중 4개, 당류 Top 5 중 3개가 스무디이고, 카테고리 평균으로도 스무디가 칼로리(457.2kcal)와 당류(49.9g) 모두 1위였다. 스무디 한 잔의 평균 당류는 WHO 하루 권고량(50g)과 거의 같다.

2. **가장 가벼운 선택은 오리지널 티 (단, 당도 0% 기준).**
   오리지널 티는 칼로리 3~5kcal, 당류 0g으로 가장 가벼웠다. 다만 홈페이지 영양정보는 당도 시럽을 넣지 않은 값으로 보이므로, 시럽을 추가하면 칼로리와 당류가 늘어난다.

3. **칼로리가 낮다고 덜 단 음료는 아님.**
   프룻티&모어는 평균 칼로리(126.7kcal)가 밀크티(252.0kcal)의 절반 수준이지만, 평균 당류(25.4g)는 오히려 밀크티(17.3g)보다 높았다. 칼로리와 당류는 따로 확인해야 한다.

4. **카페인은 커피가 가장 높고, 음료별로는 얼그레이 밀크티 계열이 상위권.**
   카테고리 평균은 커피(147mg) > 오리지널 티(97.8mg) > 밀크티(58mg) 순이었고, 음료별 Top 5에는 얼그레이 베이스 밀크티가 많았다. 반면 타로·망고·요구르트 계열 등 21개 음료(약 21%)는 카페인이 0mg이었다.

5. **토핑이 칼로리를 크게 올림.**
   같은 '딸기 쿠키 스무디'도 펄을 추가하면 587kcal에서 679kcal로 92kcal 늘어났다.

6. **베스트셀러에는 고칼로리 메뉴와 저칼로리 메뉴가 함께 있음.**
   베스트셀러 안에서 칼로리는 아메리카노 10kcal부터 딸기 쿠키 스무디 + 펄 679kcal까지 약 68배 차이가 났다. 베스트셀러 평균(291.4kcal)은 전체 평균(275kcal)보다 약간 높았다.

7. **저당 메뉴의 효과는 홈페이지 데이터만으로는 확인할 수 없었음.**
   홈페이지 수치로는 저당 메뉴와 일반 메뉴의 차이가 거의 없었다 (칼로리 +2.9kcal, 당류 0g). 이는 일반 메뉴의 영양정보가 당도 0% 기준으로 보이기 때문이며, 정확히 비교하려면 당도 옵션별 영양정보가 필요하다 (7-1 참고).

> **종합:** 공차 음료는 종류에 따라 칼로리 3kcal에서 679kcal까지 차이가 매우 크다. 칼로리와 당류를 줄이려면 스무디보다 오리지널 티나 프룻티를 고르고, 토핑을 줄이는 것이 효과적이다. 다만 홈페이지 영양정보는 당도 시럽이 빠진 값일 가능성이 높아, 실제로 주문했을 때의 당류와 칼로리는 이번 결과보다 높을 수 있다.

---

## 6. 한계점

### 6-1. 영양정보가 당도 시럽을 뺀 값일 가능성 (가장 중요)
- 일반 블랙 밀크티와 저당 블랙 밀크티의 당류가 모두 2g으로 같게 표기되어 있음
- 그런데 저당 메뉴 설명에는 "기존 블랙 밀크티 추천 당도 대비 당류 81% 감소"라고 적혀 있음
- 따라서 당도 조절이 가능한 메뉴(밀크티, 오리지널 티 등)의 영양정보는 **시럽을 넣지 않은 당도 0% 기준 값**일 가능성이 높음
- 홈페이지에 영양정보의 당도 기준이 명시되어 있지 않아 확정할 수는 없음
- 이 때문에 다음 결과는 실제로 주문했을 때와 다를 수 있음
  - **당류 0g 음료 10개:** 시럽을 넣으면 당류가 생김
  - **칼로리·당류 낮은 순위:** 오리지널 티와 밀크티가 실제보다 낮게 나왔을 수 있음
  - **저당 vs 일반 비교:** 기준이 달라 차이가 거의 없게 나옴
- 반면 브라운슈가, 스무디처럼 단맛이 음료 자체에 들어간 메뉴는 당류가 그대로 반영됨 (예: 브라운슈가 시그니처 밀크티 +미니펄 40g)
- 결과적으로 **당도 조절 메뉴와 당이 고정된 메뉴를 같은 기준으로 비교하지 못했음**

### 6-2. 분석 기준의 한계
- L 사이즈 아이스(Cold, 473ml) 기준으로만 분석함. Hot이나 J 사이즈는 값이 다름
- 납작복숭아 아이스티 3종은 L 사이즈가 없어 분석에서 제외함
- 토핑 추가, 얼음 양 같은 개인 옵션은 반영하지 못함

### 6-3. 데이터 범위의 한계
- 가격 정보가 홈페이지에 없어서 분석하지 못함
- 인기도는 베스트셀러 탭에 있는지 여부로만 판단했고, 실제 판매량은 알 수 없음
- 수집 시점의 메뉴 기준이므로 시즌 메뉴가 바뀌면 결과도 달라짐
- 일부 카테고리(오리지널 티 4개, 요거티 4개)는 음료 수가 적어서 평균이 대표성을 갖기 어려움

## 7. 추가 분석 방향

### 7-1. 당도 옵션별 영양정보 확보 (최우선)
한계점에서 확인했듯이, 홈페이지 영양정보는 당도 시럽을 넣지 않은 값일 가능성이 높다. 당도(0%, 30%, 50%, 70%, 100%)별로 시럽이 더하는 당류와 칼로리를 공식 앱, 매장, 고객센터 문의 등으로 확인할 수 있다면 다음 분석을 다시 할 수 있다.
- 추천 당도 기준으로 칼로리·당류 순위를 다시 매겨, 실제 주문 상태에 가까운 결과 만들기
- 저당 메뉴와 일반 메뉴를 같은 기준(추천 당도)으로 비교해 '81% 감소'가 실제 수치로 어느 정도인지 확인하기
- 당도 조절 메뉴와 당이 고정된 메뉴(브라운슈가, 스무디 등)를 공정하게 비교하기

### 7-2. 토핑 영양정보 확보
펄, 치즈폼, 밀크폼 같은 토핑은 메뉴 이름에 포함된 경우에만 영양정보에 반영되어 있다. 이번 분석에서 '딸기 쿠키 스무디'에 펄을 넣으면 92kcal가 늘어나는 것을 확인했는데, 토핑별 영양정보를 따로 찾으면 토핑 추가에 따른 변화를 모든 메뉴에 대해 계산할 수 있다.

### 7-3. 가격 정보 수집
배달 앱이나 공식 주문 앱에서 메뉴별 가격을 수집하면 '카테고리별 평균 가격', '가격 대비 칼로리', '비쌀수록 칼로리가 높은가' 같은 분석을 추가할 수 있다.

### 7-4. 인기도 데이터 수집
배달 앱의 리뷰 수나 주문 순위처럼 인기를 숫자로 나타내는 데이터를 찾으면, 소비자가 실제로 많이 고르는 음료의 영양 성분 특징을 분석할 수 있다. 이번에는 베스트셀러의 평균 칼로리가 전체 평균보다 약간 높았는데, 판매량 데이터로 이 경향을 더 정확히 확인할 수 있다.

### 7-5. 다른 브랜드와 비교
이디야, 메가커피, 컴포즈커피 등 다른 브랜드의 공식 영양정보를 같은 방식으로 수집해 밀크티, 아메리카노처럼 비슷한 메뉴끼리 비교할 수 있다. 이때 브랜드마다 영양정보의 당도 기준과 컵 용량이 다를 수 있으므로, 같은 기준으로 맞춘 뒤 비교해야 한다. 수집 전에는 각 사이트의 robots.txt를 확인해야 한다.

### 7-6. 시간에 따른 메뉴 변화
같은 코드를 주기적으로 실행해 데이터를 쌓으면, 시즌 신메뉴의 칼로리와 당류가 시간에 따라 어떻게 변하는지 추적할 수 있다. 과거 메뉴 정보는 공식 SNS나 보도자료에서 찾을 수 있다.
